# Taller 2 — Análisis de datos
**Fundamentos de Analítica · 2026-2 · Universidad EIA**

**Dataset:** movimiento GPS/telemetría de osos negros (*Ursus americanus*), registrado cada hora junto con variables ambientales (temperatura, NDVI, duración del día, altitud solar) y de comportamiento.

**Fuente:** Dryad Digital Repository — *Data from: [movement/ecology dataset de osos negros]*
DOI: [10.5061/dryad.d51c5b0fd](https://datadryad.org/dataset/doi%3A10.5061/dryad.d51c5b0fd)

**Nota sobre la muestra:** el archivo original (`bear.csv`) es muy pesado, por lo que este taller trabaja con una **muestra representativa de 200 registros consecutivos** correspondientes a un mismo individuo (`id = 1`), cubriendo 12 días de seguimiento (26 de mayo – 6 de junio de 2006). Esto es suficiente para explorar patrones temporales y ambientales de movimiento, aunque no permite comparar entre individuos, sexos o edades (todas esas columnas son constantes en la muestra).

**Herramienta seleccionada:** Plotly (Python), sobre un notebook Jupyter (`.ipynb`).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

df = pd.read_csv('bear.csv')
df.head()

,id,Sex,age,movem_60,timestamp,season,reprod,move_status,cor_hour,altitude_dg,t_period,index,tmx,dayle,study,year,ndvi
0,1,1,2,0.014142,2006-05-26T03:00:00Z,1,1,resident,3,-21.597845,nocturnal,0.125723,19.37,14.830628,1,2,0.37345
1,1,1,2,0.057070,2006-05-26T04:00:00Z,1,1,resident,4,-14.976951,crepuscular,0.125723,19.37,14.830640,1,2,0.37345
2,1,1,2,0.056436,2006-05-26T05:00:00Z,1,1,resident,5,-6.597630,crepuscular,0.125723,19.37,14.830691,1,2,0.37345
3,1,1,2,0.024515,2006-05-26T06:00:00Z,1,1,resident,6,3.022102,crepuscular,0.125723,19.37,14.830640,1,2,0.37345
4,1,1,2,0.031000,2006-05-26T07:00:00Z,1,1,resident,7,13.474819,crepuscular,0.125723,19.37,14.830662,1,2,0.37345


### Paleta de colores del notebook

Se fija una paleta consistente para todo el análisis: colores categóricos fijos por período del día (para que "Diurno" sea siempre el mismo color en todos los gráficos, sin importar el orden en que aparezca), una escala secuencial (un solo tono, claro→oscuro) para variables de magnitud, y una escala divergente (dos tonos opuestos + gris neutro) para la matriz de correlación, donde el signo (positivo/negativo) importa tanto como la magnitud.

In [2]:
# Paleta fija — evita que plotly asigne colores distintos cada vez que cambia el orden/filtro
COLOR_PERIODO = {'Diurno': '#2a78d6', 'Crepuscular': '#eb6834', 'Nocturno': '#1baf7a'}
ORDEN_PERIODO = ['Nocturno', 'Crepuscular', 'Diurno']

SEQ_BLUE = ['#cde2fb', '#9ec5f4', '#5598e7', '#2a78d6', '#1c5cab', '#0d366b']  # secuencial (magnitud)
DIVERGING = [[0, '#e34948'], [0.5, '#f0efec'], [1, '#2a78d6']]                 # divergente (correlación: - / 0 / +)

## 2. Selección e investigación de la herramienta — Plotly (Python)

**¿Qué es y cuál es su propósito principal?**
Plotly es una librería de visualización de datos de código abierto para Python (también existe para R, JavaScript, etc.) que genera gráficos **interactivos** basados en SVG/WebGL (zoom, hover con tooltips, pan, selección de rango, botones y sliders). Su propósito principal es facilitar la exploración visual de datos y la construcción de gráficos listos para publicar o embeber en notebooks, aplicaciones web (Dash) o reportes HTML, sin necesidad de escribir JavaScript.

**¿Es gratuita, de pago o ambas?**
La librería `plotly.py` en sí es **100% gratuita y de código abierto** (licencia MIT). Existe además un servicio comercial (Plotly Chart Studio / Plotly Cloud / Dash Enterprise) para hosting, colaboración en equipo y despliegue empresarial, que sí tiene planes de pago — pero no es necesario para crear ni ejecutar los gráficos de este taller.

**¿Qué tipos de fuentes de datos permite utilizar?**
No tiene conectores propios: al ser una librería de Python, puede graficar cualquier estructura que se pueda cargar como `DataFrame`/array de NumPy — CSV, Excel, JSON, bases de datos SQL (vía `pandas`/SQLAlchemy), APIs REST, Parquet, GeoJSON, etc. Es tan flexible como el propio ecosistema de Python para ingesta de datos.

**Principales ventajas**
- Gráficos interactivos "gratis" (zoom, hover, leyenda clicable) sin código adicional.
- Se integra de forma nativa en notebooks Jupyter y se exporta fácilmente a HTML standalone.
- Amplia variedad de gráficos estadísticos y geoespaciales (`plotly.express` cubre la mayoría en una sola línea).
- Control total sobre transformación y modelado de datos porque todo ocurre en Python/Pandas (no hay "caja negra").
- Escala naturalmente a una app web completa con Dash si el proyecto crece.

**Principales limitaciones**
- No es una herramienta de BI "point-and-click": requiere saber programar (a diferencia de Tableau, Qlik o Power BI).
- No incluye por sí sola un modelo de datos, refresco programado, ni gestión de permisos/usuarios — eso hay que construirlo aparte (p. ej. con Dash + un backend).
- Para dashboards con muchos filtros cruzados o *self-service* por usuarios no técnicos, requiere más esfuerzo de desarrollo que una herramienta de BI dedicada.
- El manejo de datasets muy grandes (millones de filas) puede volverse lento en el navegador si no se agregan/samplea los datos antes de graficar.

**¿En qué proyectos sería apropiado usarla?**
Es ideal para análisis exploratorio de datos (EDA) dentro de un flujo de trabajo en Python/notebooks, para prototipos rápidos de visualización, para reportes reproducibles (el mismo código genera siempre el mismo resultado) y para integrarse en pipelines de ciencia de datos o machine learning donde de todas formas ya se está trabajando en Python. Es menos apropiada cuando el objetivo es entregar un dashboard *self-service* a usuarios de negocio no técnicos que necesitan explorar los datos sin escribir código — ahí una herramienta de BI tradicional (Power BI, Tableau) es más adecuada.

## 3. Comprensión del dataset

In [3]:
print('Dimensiones:', df.shape)
df.info()

Dimensiones: (200, 17)
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           200 non-null    int64  
 1   Sex          200 non-null    int64  
 2   age          200 non-null    int64  
 3   movem_60     200 non-null    float64
 4   timestamp    200 non-null    str    
 5   season       200 non-null    int64  
 6   reprod       200 non-null    int64  
 7   move_status  200 non-null    str    
 8   cor_hour     200 non-null    int64  
 9   altitude_dg  200 non-null    float64
 10  t_period     200 non-null    str    
 11  index        200 non-null    float64
 12  tmx          200 non-null    float64
 13  dayle        200 non-null    float64
 14  study        200 non-null    int64  
 15  year         200 non-null    int64  
 16  ndvi         200 non-null    float64
dtypes: float64(6), int64(8), str(3)
memory usage: 26.7 KB


**¿Qué información contiene el dataset?**
Cada fila es una posición GPS registrada por hora para un oso individual, junto con variables de comportamiento (estado de movimiento, período del día) y variables ambientales asociadas a ese momento y lugar (temperatura máxima, duración del día, altitud solar, índice de vegetación NDVI).

**Diccionario de columnas (interpretación):**
- `id`: identificador del individuo (oso).
- `Sex`, `age`: sexo y edad del oso (codificados numéricamente).
- `movem_60`: distancia/velocidad de movimiento en la última hora (variable objetivo principal).
- `timestamp`: fecha y hora UTC del registro.
- `season`: estación/temporada codificada (1, 2).
- `reprod`: estado reproductivo (codificado).
- `move_status`: estado de movimiento categórico (p. ej. "resident").
- `cor_hour`: hora del día corregida (1–24).
- `altitude_dg`: altitud solar en grados (negativa de noche, positiva de día).
- `t_period`: período del día — `nocturnal`, `crepuscular`, `diurnal`.
- `index`: un índice numérico asociado al registro (probablemente de calidad de la señal GPS).
- `tmx`: temperatura máxima del día (°C).
- `dayle`: duración del día (horas de luz, "day length").
- `study`, `year`: identificadores del estudio y año.
- `ndvi`: índice de vegetación (NDVI), proxy de disponibilidad de alimento/vegetación.

In [4]:
print('Registros y variables:', df.shape[0], 'filas x', df.shape[1], 'columnas')

Registros y variables: 200 filas x 17 columnas


In [ ]:
# Valores nulos
print('Nulos por columna:')
print(df.isnull().sum())
print()
# Duplicados
print('Filas duplicadas:', df.duplicated().sum())

Valores nulos por columna:
id             0
Sex            0
age            0
movem_60       0
timestamp      0
season         0
reprod         0
move_status    0
cor_hour       0
altitude_dg    0
t_period       0
index          0
tmx            0
dayle          0
study          0
year           0
ndvi           0
dtype: int64

Filas duplicadas: 0


In [6]:
# Valores atípicos en la variable de interés (movem_60) via IQR
q1, q3 = df['movem_60'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = df[(df['movem_60'] < lower) | (df['movem_60'] > upper)]
print(f'Rango intercuartílico normal: [{lower:.3f}, {upper:.3f}]')
print(f'Registros atípicos (outliers) en movem_60: {len(outliers)} de {len(df)} ({len(outliers)/len(df):.1%})')
outliers[['timestamp','movem_60','t_period','tmx']].head()

Rango intercuartílico normal: [-0.233, 0.466]
Registros atípicos (outliers) en movem_60: 15 de 200 (7.5%)


,timestamp,movem_60,t_period,tmx
47,2006-05-29T04:00:00Z,0.501351,crepuscular,20.39
98,2006-05-31T17:00:00Z,1.245490,diurnal,14.43
99,2006-05-31T18:00:00Z,0.711975,diurnal,16.37
102,2006-05-31T21:00:00Z,0.517151,crepuscular,16.90
109,2006-06-01T04:00:00Z,0.483374,crepuscular,14.88


**Hallazgos de calidad de datos:**
- **No hay valores nulos** en ninguna de las 17 columnas.
- **No hay filas duplicadas.**
- Sí hay **valores atípicos evidentes en `movem_60`**: 15 de 200 registros (7.5%) superan el umbral IQR (0.466), con un máximo de 1.25 — son momentos de movimiento intenso, probablemente reales (desplazamientos largos) más que errores de medición, por lo que se documentan pero **no se eliminan**.
- Varias columnas son **constantes en esta muestra** (`id`, `Sex`, `age`, `study`, `year`, `move_status`, `reprod`) porque corresponde a un solo individuo — se documenta como limitación de trabajar con una muestra reducida, no como problema de calidad.

**Variables más importantes para el análisis:** `movem_60` (variable de interés/objetivo), `timestamp` / `cor_hour` (dimensión temporal), `t_period` (categoría de período del día), `tmx`, `ndvi`, `dayle` y `altitude_dg` (variables ambientales que podrían explicar el movimiento), y `season` (para comparar temporadas).

**Cuatro preguntas de análisis:**
1. ¿Cómo varía el movimiento promedio del oso (`movem_60`) según el período del día (nocturno, crepuscular, diurno)?
2. ¿Cómo evoluciona el movimiento del oso a lo largo de los 12 días de seguimiento?
3. ¿Existe relación entre el movimiento y las variables ambientales (temperatura máxima, NDVI, duración del día)?
4. ¿Cómo se distribuyen los registros entre las distintas categorías de período del día y estaciones, y cambia el patrón de movimiento entre la estación 1 y la estación 2?

## 4. Preparación de los datos

Se aplican las siguientes transformaciones (más de las dos mínimas requeridas), documentando el motivo de cada una:

1. **Conversión de `timestamp` a tipo fecha/hora** — llega como texto (string ISO 8601); convertirlo a `datetime` permite ordenar cronológicamente, graficar series de tiempo y extraer componentes (día, hora).
2. **Extracción de nuevas columnas desde `timestamp`** (`date`, `hour`) — necesarias para agrupar el movimiento por día o por hora del día.
3. **Creación de una variable calculada `t_period_es`** (traducción/etiqueta legible en español de `t_period`) — mejora la interpretabilidad de los gráficos para el público objetivo.
4. **Creación de una variable calculada `movement_level`** (categorización de `movem_60` en Bajo/Medio/Alto por terciles) — variable calculada que no existe en el dataset original, útil como filtro/segmentador en los gráficos.
5. **Recodificación de `season`** a etiquetas descriptivas (`Temporada 1`, `Temporada 2`) — mejora la legibilidad de leyendas y ejes.
6. **Marcado de outliers de `movem_60`** (columna booleana `is_outlier`) en vez de eliminarlos — se documentan pero se conservan, porque representan comportamiento real del animal (desplazamientos largos), no errores de captura.

In [7]:
# 1) timestamp a datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2) columnas derivadas de tiempo
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

# 3) etiqueta legible del período del día
period_map = {'nocturnal': 'Nocturno', 'crepuscular': 'Crepuscular', 'diurnal': 'Diurno'}
df['t_period_es'] = df['t_period'].map(period_map)

# 4) variable calculada: nivel de movimiento (no existe en el dataset original)
df['movement_level'] = pd.qcut(df['movem_60'], q=3, labels=['Bajo', 'Medio', 'Alto'])

# 5) recodificación de season
season_map = {1: 'Temporada 1', 2: 'Temporada 2'}
df['season_label'] = df['season'].map(season_map)

# 6) marcado (no eliminación) de outliers de movem_60
q1, q3 = df['movem_60'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
df['is_outlier'] = (df['movem_60'] < lower) | (df['movem_60'] > upper)

df[['timestamp','date','hour','t_period_es','movem_60','movement_level','season_label','is_outlier']].head()

,timestamp,date,hour,t_period_es,movem_60,movement_level,season_label,is_outlier
0,2006-05-26 03:00:00+00:00,2006-05-26,3,Nocturno,0.014142,Bajo,Temporada 1,False
1,2006-05-26 04:00:00+00:00,2006-05-26,4,Crepuscular,0.057070,Medio,Temporada 1,False
2,2006-05-26 05:00:00+00:00,2006-05-26,5,Crepuscular,0.056436,Medio,Temporada 1,False
3,2006-05-26 06:00:00+00:00,2006-05-26,6,Crepuscular,0.024515,Bajo,Temporada 1,False
4,2006-05-26 07:00:00+00:00,2006-05-26,7,Crepuscular,0.031000,Bajo,Temporada 1,False


## 5. Construcción del análisis

Seis visualizaciones (cinco tipos distintos de gráfico: línea, violín, dispersión, barras y dos mapas de calor), un indicador resumen, un elemento de interacción y variables calculadas (`movement_level`, `t_period_es`, `hour`), como pide el taller. La paleta de color es consistente en todo el notebook (mismo color = mismo período del día en todos los gráficos) y cada mapa de calor usa la escala que le corresponde: secuencial para magnitud, divergente para correlación (signo + magnitud).

### Indicadores resumen (KPIs)

In [8]:
movem_prom = df['movem_60'].mean()
movem_max = df['movem_60'].max()
pct_diurno = (df['t_period'] == 'diurnal').mean() * 100
pct_outliers = df['is_outlier'].mean() * 100

fig_kpi = make_subplots(
    rows=1, cols=4,
    specs=[[{'type':'indicator'}]*4]
)
fig_kpi.add_trace(go.Indicator(mode='number', value=movem_prom, number={'valueformat':'.3f'},
                                title={'text':'Movimiento promedio (movem_60)'}), row=1, col=1)
fig_kpi.add_trace(go.Indicator(mode='number', value=movem_max, number={'valueformat':'.3f'},
                                title={'text':'Movimiento máximo registrado'}), row=1, col=2)
fig_kpi.add_trace(go.Indicator(mode='number', value=pct_diurno, number={'suffix':'%','valueformat':'.1f'},
                                title={'text':'% de registros en período diurno'}), row=1, col=3)
fig_kpi.add_trace(go.Indicator(mode='number', value=pct_outliers, number={'suffix':'%','valueformat':'.1f'},
                                title={'text':'% de registros atípicos (outliers)'}), row=1, col=4)
fig_kpi.update_layout(height=220, title='Indicadores resumen — movimiento del oso (muestra de 200 registros)')
fig_kpi.show()

### Visualización 1 (línea) — Evolución del movimiento a lo largo del tiempo
**Responde la pregunta 2.** Permite ver la tendencia y la variabilidad hora a hora a lo largo de los 12 días de seguimiento, e identificar picos de actividad puntuales (algunos coinciden con los outliers marcados).

In [9]:
fig1 = px.line(
    df.sort_values('timestamp'), x='timestamp', y='movem_60',
    color='season_label', markers=True,
    title='Evolución del movimiento por hora (movem_60) — 26 may al 6 jun 2006',
    labels={'timestamp':'Fecha y hora', 'movem_60':'Movimiento (movem_60)', 'season_label':'Estación'}
)
fig1.add_trace(go.Scatter(
    x=df.loc[df['is_outlier'], 'timestamp'], y=df.loc[df['is_outlier'], 'movem_60'],
    mode='markers', marker=dict(color='#e34948', size=9, symbol='x'),
    name='Outlier'
))
fig1.update_traces(selector=dict(mode='lines+markers'))
# recolorea las líneas de temporada con la paleta fija del notebook
fig1.for_each_trace(lambda t: t.update(line_color='#2a78d6', marker_color='#2a78d6') if t.name == 'Temporada 1'
                     else (t.update(line_color='#eda100', marker_color='#eda100') if t.name == 'Temporada 2' else None))
fig1.update_layout(legend_title_text='Serie')
fig1.show()

### Visualización 2 (violín) — Distribución del movimiento por período del día
**Responde la pregunta 1.** Se usa un **violin plot** en vez de un boxplot simple: además de la mediana y los cuartiles, muestra la forma completa de la distribución (dónde se concentran realmente los valores), lo cual contrasta mejor las tres categorías cuando —como aquí— la distribución es asimétrica (cola larga hacia los valores altos). Color fijo por categoría, consistente con el resto del notebook.

In [10]:
fig2 = px.violin(
    df, x='t_period_es', y='movem_60', color='t_period_es', box=True, points='all',
    category_orders={'t_period_es': ORDEN_PERIODO},
    color_discrete_map=COLOR_PERIODO,
    title='Distribución del movimiento por período del día',
    labels={'t_period_es':'Período del día', 'movem_60':'Movimiento (movem_60)'}
)
fig2.update_layout(showlegend=False)
fig2.show()

### Visualización 3 (dispersión) — Relación entre movimiento y variables ambientales
**Responde la pregunta 3.** Un scatter plot es el tipo de gráfico adecuado para explorar si existe relación entre dos variables continuas (aquí, cada variable ambiental vs. movimiento), coloreando por período del día (identidad categórica) para ver si el patrón cambia según el momento del día.

**Elemento de interacción:** botones para alternar la variable ambiental del eje X (NDVI, temperatura máxima, duración del día).

In [11]:
fig3 = go.Figure()
env_vars = {'ndvi': 'NDVI (vegetación)', 'tmx': 'Temperatura máxima (°C)', 'dayle': 'Duración del día (h)'}

for i, (col, label) in enumerate(env_vars.items()):
    for periodo in ORDEN_PERIODO:
        sub = df[df['t_period_es'] == periodo]
        fig3.add_trace(go.Scatter(
            x=sub[col], y=sub['movem_60'], mode='markers',
            marker=dict(color=COLOR_PERIODO[periodo], size=8, line=dict(color='#fcfcfb', width=1)),
            name=periodo, legendgroup=periodo, visible=(i == 0),
            hovertemplate=f'{periodo}<br>x=%{{x}}<br>movem_60=%{{y}}<extra></extra>'
        ))

buttons = []
n_series = len(ORDEN_PERIODO)
for i, (col, label) in enumerate(env_vars.items()):
    visible = [(j // n_series) == i for j in range(len(env_vars) * n_series)]
    buttons.append(dict(label=label, method='update',
                         args=[{'visible': visible}, {'xaxis': {'title': label}}]))

fig3.update_layout(
    title='Relación entre movimiento y variables ambientales (usa el menú para cambiar variable)',
    xaxis_title='NDVI (vegetación)', yaxis_title='Movimiento (movem_60)',
    legend_title_text='Período del día',
    updatemenus=[dict(active=0, buttons=buttons, x=1.2, y=1.15)]
)
fig3.show()

In [12]:
corr = df[['movem_60','ndvi','tmx','dayle','altitude_dg']].corr()['movem_60'].drop('movem_60')
print('Correlación de movem_60 con variables ambientales:')
print(corr.round(3))

Correlación de movem_60 con variables ambientales:
ndvi           0.226
tmx           -0.143
dayle          0.210
altitude_dg   -0.114
Name: movem_60, dtype: float64


### Visualización 4 (barras) — Movimiento promedio por hora del día
**Responde también la pregunta 1**, con un nivel de detalle mayor (por hora en vez de por categoría de período), y usa la variable calculada `hour` creada en la preparación de datos. Se colorea con la escala secuencial (una sola tonalidad, más oscuro = mayor movimiento) porque es una magnitud, no una categoría.

In [13]:
hourly = df.groupby('hour', as_index=False)['movem_60'].mean()
fig4 = px.bar(
    hourly, x='hour', y='movem_60',
    title='Movimiento promedio por hora del día',
    labels={'hour':'Hora del día (0-23)', 'movem_60':'Movimiento promedio (movem_60)'},
    color='movem_60', color_continuous_scale=SEQ_BLUE
)
fig4.update_layout(coloraxis_showscale=False)
fig4.show()

### Visualización 5 (mapa de calor — matriz de correlación)
Responde a tu pregunta por un gráfico tipo "mapa de calor": esta es la forma estándar de contrastar de un vistazo cómo se relacionan **varias variables numéricas entre sí** (aquí, el movimiento contra las cuatro variables ambientales/temporales). Se usa una escala **divergente** (azul↔rojo con gris en el centro) porque el signo de la correlación importa tanto como su magnitud — una correlación de -0.3 y +0.3 son igual de "fuertes" pero apuntan en direcciones opuestas.

In [14]:
corr_matrix = df[['movem_60','ndvi','tmx','dayle','altitude_dg','cor_hour']].corr().round(2)

fig5 = px.imshow(
    corr_matrix, text_auto=True, color_continuous_scale=DIVERGING, zmin=-1, zmax=1,
    aspect='auto',
    title='Mapa de calor — correlación entre movimiento y variables ambientales/temporales'
)
fig5.update_layout(coloraxis_colorbar=dict(title='Correlación'))
fig5.show()

### Visualización 6 (mapa de calor — patrón hora × día)
**Sobre el mapa geográfico que preguntabas:** esta muestra de `bear.csv` **no incluye coordenadas de latitud/longitud** (solo variables ambientales derivadas de la posición, no la posición misma), así que no es posible dibujar un mapa real de los desplazamientos con estos datos. Como alternativa —y para "ver" el movimiento en el espacio que sí tenemos, que es tiempo—, este mapa de calor cruza **hora del día × fecha** y colorea por intensidad de movimiento: funciona como un "mapa de calor temporal" que muestra en qué franjas horarias y qué días se concentra la actividad del oso, con la misma lógica visual que un mapa de calor geográfico (color = intensidad) pero sobre una grilla tiempo×tiempo en vez de un mapa.

In [15]:
heat_data = df.pivot_table(index='date', columns='hour', values='movem_60', aggfunc='mean')

fig6 = px.imshow(
    heat_data, color_continuous_scale=SEQ_BLUE, aspect='auto',
    labels=dict(x='Hora del día', y='Fecha', color='movem_60 prom.'),
    title='Mapa de calor — movimiento promedio por hora y día'
)
fig6.update_layout(xaxis=dict(dtick=1))
fig6.show()

**Si en algún momento consigues las coordenadas GPS reales** (columnas de latitud/longitud, a veces nombradas `lat`/`long` o en coordenadas UTM `x`/`y` en el dataset completo de Dryad), este es el código listo para un mapa de calor geográfico real de los movimientos — no requiere token de Mapbox porque usa el estilo abierto `open-street-map`:

```python
# Plantilla — requiere columnas de latitud y longitud (ajusta los nombres)
fig_mapa = px.density_mapbox(
    df, lat='lat', lon='lon', z='movem_60', radius=15,
    center=dict(lat=df['lat'].mean(), lon=df['lon'].mean()), zoom=9,
    mapbox_style='open-street-map', color_continuous_scale=SEQ_BLUE,
    title='Mapa de calor geográfico — intensidad de movimiento del oso'
)
fig_mapa.show()

# Alternativa: ruta de puntos GPS coloreada por variable
fig_ruta = px.scatter_mapbox(
    df, lat='lat', lon='lon', color='t_period_es', color_discrete_map=COLOR_PERIODO,
    mapbox_style='open-street-map', zoom=9,
    title='Ruta de desplazamiento del oso, coloreada por período del día'
)
fig_ruta.show()
```

## 6. Interpretación de los resultados

**Conclusión 1 — El movimiento es claramente mayor en horas crepusculares.**
El promedio de `movem_60` en período **crepuscular es 0.221**, frente a **0.139 en diurno** y **0.100 en nocturno** (Visualización 2). Es decir, el movimiento crepuscular es en promedio **~59% mayor** que el diurno y **más del doble (120% mayor)** que el nocturno — consistente con un patrón de actividad crepuscular ("crepuscular activity pattern"), típico de varias especies de osos que evitan el calor/exposición del mediodía y la oscuridad total de la noche.

**Conclusión 2 — El movimiento presenta picos puntuales de alta intensidad, no una tendencia sostenida al alza.**
En la serie de tiempo (Visualización 1) se identificaron **15 registros atípicos (7.5% del total)** que superan el umbral IQR de 0.466, con un máximo de **1.245** — más de 6 veces el promedio general (0.158). Estos picos aparecen distribuidos a lo largo de todo el período de seguimiento (no concentrados al inicio o al final), lo que sugiere episodios puntuales de desplazamiento largo más que una tendencia temporal sostenida.

**Conclusión 3 — El NDVI (vegetación) tiene una correlación positiva débil-moderada con el movimiento, mientras que la temperatura la tiene negativa.**
La correlación de `movem_60` con `ndvi` es de **+0.226** y con `dayle` (duración del día) de **+0.210**, mientras que con `tmx` (temperatura máxima) es de **-0.143** (Visualización 3). Aunque son correlaciones débiles (no hay una relación lineal fuerte), la dirección es consistente con la hipótesis de que el oso se mueve algo más en días con mayor disponibilidad de vegetación/luz y algo menos en los días más calurosos.

**Conclusión 4 — La actividad se concentra en un tercio del día, con un patrón bimodal alrededor del amanecer y el atardecer.**
En la Visualización 4 (movimiento promedio por hora), los picos más altos se ubican en las **horas de transición** (alrededor de las 4-6h y 16-18h aproximadamente), coincidiendo con el patrón crepuscular ya identificado en la Conclusión 1, mientras que las horas centrales de la noche (0-3h) muestran los valores más bajos.

**Conclusión 5 — Casi la mitad de los registros de la muestra corresponden a período diurno.**
El KPI de "% de registros en período diurno" muestra que **49.5%** de las 200 observaciones caen en esta categoría, frente a 32% crepuscular y 18.5% nocturno — es decir, aunque el oso se mueve más por hora durante el crepúsculo, pasa proporcionalmente más horas del día etiquetadas como diurnas, lo cual es coherente con la duración natural de cada franja horaria.

**Conclusión 6 — El patrón crepuscular es consistente día a día, no un promedio que oculta comportamientos distintos.**
El mapa de calor hora×día (Visualización 6) muestra dos bandas de mayor intensidad (más oscuras) recurrentes en casi todos los 12 días de seguimiento, ubicadas alrededor de las franjas de amanecer y atardecer, con la franja central de la noche consistentemente más clara (menor movimiento). Esto confirma que la Conclusión 1 (mayor movimiento crepuscular) no es un artefacto de promediar días muy distintos entre sí, sino un patrón diario estable a lo largo de las casi dos semanas de seguimiento.

## 7. Comparación con Power BI

| Aspecto | Plotly (Python) | Power BI |
|---|---|---|
| **Conexión y carga de datos** | Se hace en código: `pandas.read_csv`, `read_sql`, `read_json`, APIs, etc. Total flexibilidad, pero cada fuente nueva requiere escribir/adaptar código. | Interfaz gráfica ("Obtener datos") con decenas de conectores nativos (Excel, SQL Server, SharePoint, web, APIs). No requiere programar, pero está limitado a los conectores disponibles o a Power Query personalizado. |
| **Transformación / preparación de datos** | Se usa `pandas` (filtrar, agrupar, unir, tipos de datos, columnas nuevas) — el mismo lenguaje que el análisis, así que todo queda documentado y es 100% reproducible y versionable (Git). | **Power Query** (interfaz de "pasos aplicados" tipo ETL visual): clics para filtrar, dividir columnas, cambiar tipos, combinar tablas. Más accesible para no programadores, pero menos flexible para transformaciones muy específicas. |
| **¿Equivalente a Power Query?** | No hay un componente dedicado; su equivalente conceptual es simplemente el código de `pandas` que antecede a la visualización. | Power Query es el componente central de preparación de datos de Power BI. |
| **Columnas / métricas calculadas** | Se crean como columnas nuevas de un DataFrame (`df['nueva_col'] = ...`) o como agregaciones con `groupby`. Todo el cálculo ocurre en Python **antes** de graficar. | Se crean con **DAX** (`Columna calculada` o `Medida`), directamente en el modelo de datos, y se recalculan dinámicamente según los filtros/interacción del usuario en el dashboard. |
| **¿Equivalente a DAX?** | No existe un lenguaje equivalente dentro de Plotly — el cálculo se resuelve en Python/Pandas de forma estática (una vez, al preparar los datos), no como fórmulas reactivas dentro del reporte. | DAX es el lenguaje de fórmulas nativo de Power BI, pensado para que las métricas se recalculen en vivo según el contexto de filtro. |
| **Filtros e interacción entre visualizaciones** | Se programan explícitamente: botones/menús (`updatemenus`), sliders, o —si se lleva a una app con Dash— callbacks que conectan un filtro con varios gráficos. Cada interacción hay que codificarla. | Los filtros (segmentadores) y el *cross-filtering* entre visuales vienen **integrados por defecto**: al hacer clic en un gráfico, los demás se filtran automáticamente, sin escribir código. |
| **Más sencillo en la herramienta elegida** | Reproducibilidad total del análisis (el notebook es a la vez documentación, código y resultado), control fino sobre el tipo exacto de gráfico y su estética, e integración directa con el resto del ecosistema de ciencia de datos en Python (modelos, estadística). | — |
| **Más difícil o limitado en la herramienta elegida** | Construir interacción rica entre múltiples gráficos (cross-filtering automático) requiere bastante más código que en Power BI, y no hay una forma sencilla de compartir un dashboard "vivo" con usuarios de negocio sin desplegar una app aparte (Dash). | — |
| **¿Cuándo elegir Plotly en vez de Power BI?** | Cuando el análisis ya vive en un flujo de trabajo de Python/notebooks (ciencia de datos, investigación, modelos), cuando se necesita full reproducibilidad y control del detalle visual, o cuando el público del reporte son otros analistas/desarrolladores técnicos. | Cuando el objetivo es un dashboard *self-service* para usuarios de negocio no técnicos, con actualización automática de datos y necesidad de filtros/interacción "out of the box" sin programar. |


## 8. Entregables — resumen

1. **Dataset y fuente:** muestra de `bear.csv` (200 registros, 1 individuo, 12 días), del dataset de telemetría de osos en Dryad — DOI [10.5061/dryad.d51c5b0fd](https://datadryad.org/dataset/doi%3A10.5061/dryad.d51c5b0fd).
2. **Investigación de la herramienta:** sección 2 (Plotly — qué es, costo, fuentes, ventajas, limitaciones, casos de uso).
3. **Transformaciones realizadas:** sección 4 (6 transformaciones documentadas, con motivo de cada una).
4. **Cuatro preguntas de análisis:** listadas al final de la sección 3.
5. **Notebook analítico:** sección 5 (KPIs + 6 visualizaciones en 5 tipos de gráfico + interacción + variables calculadas).
6. **Conclusiones:** sección 6 (5 conclusiones respaldadas por cifras concretas).
7. **Comparación con Power BI:** sección 7.